<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_09_1_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-558: Applications of Deep Neural Networks
**Module 9: Foundations of Generative AI**

* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).

# Module 9 Material

* **Part 9.1: Transformer Architecture** [[Video]]() [[Notebook]](t81_558_class_09_1_transformers.ipynb)
* Part 9.2: Pretrained Models and the Hugging Face Ecosystem [[Video]]() [[Notebook]](t81_558_class_09_2_hugging.ipynb)
* Part 9.3: Embeddings and Vector Representations [[Video]]() [[Notebook]](t81_558_class_09_3_embedding.ipynb)
* Part 9.4: Diffusion Models and Image Generation [[Video]]() [[Notebook]](t81_558_class_09_4_stable_diff.ipynb)
* Part 9.5: Accessing API-Based LLMs [[Video]]() [[Notebook]](t81_558_class_09_5_chat_gpt.ipynb)


# Google CoLab Instructions

The following code ensures that Google CoLab is running the correct version of TensorFlow.
  Running the following code will map your GDrive to ```/content/drive```.

In [ ]:
try:
    from google.colab import drive
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

Note: using Google CoLab


# Part 9.1: Introduction to Transformers

Transformers are neural networks that provide state-of-the-art solutions for many of the problems previously assigned to recurrent neural networks. [[Cite:vaswani2017attention]](https://arxiv.org/abs/1706.03762) Sequences can form both the input and the output of a neural network, examples of such configurations include::

* Vector to Sequence - Image captioning
* Sequence to Vector - Sentiment analysis
* Sequence to Sequence - Language translation

Sequence-to-sequence allows an input sequence to produce an output sequence based on an input sequence. Transformers focus primarily on this sequence-to-sequence configuration.

## High-Level Overview of Transformers

This course focuses primarily on the application of deep neural networks. The focus will be on presenting data to a transformer and a transformer's major components. As a result, we will not focus on implementing a transformer at the lowest level. The following section provides an overview of critical internal parts of a transformer, such as residual connections and attention. In the next chapter, we will use transformers from [Hugging Face](https://huggingface.co/) to perform natural language processing with transformers. If you are interested in implementing a transformer from scratch, Keras provides a comprehensive [example](https://www.tensorflow.org/text/tutorials/transformer).

Figure 10.TRANS-1 presents a high-level view of a transformer for language translation.

**Figure 10.TRANS-1: High Level View of a Translation Transformer**
![Transformer](https://data.heatonresearch.com/images/jupyter/transformer-1.jpg)

We use a transformer that translates between English and Spanish for this example. We present the English sentence "the cat likes milk" and receive a Spanish translation of "al gato le gusta la leche."

We begin by placing the English source sentence between the beginning and ending tokens. This input can be of any length, and we presented it to the neural network as a ragged Tensor. Because the Tensor is ragged, no padding is necessary. Such input is acceptable for the attention layer that will receive the source sentence. The encoder transforms this ragged input into a hidden state containing a series of key-value pairs representing the knowledge in the source sentence. The encoder understands to read English and convert to a hidden state. The decoder understands how to output Spanish from this hidden state.

We initially present the decoder with the hidden state and the starting token. The decoder will predict the probabilities of all words in its vocabulary. The word with the highest probability is the first word of the sentence.

The highest probability word is attached concatenated to the translated sentence, initially containing only the beginning token. This process continues, growing the translated sentence in each iteration until the decoder predicts the ending token.

## Transformer Hyperparameters

Before we describe how these layers fit together, we must consider the following transformer hyperparameters, along with default settings from the Keras transformer example:

* num_layers = 4
* d_model = 128
* dff = 512
* num_heads = 8
* dropout_rate = 0.1

Multiple encoder and decoder layers can be present. The **num_layers** hyperparameter specifies how many encoder and decoder layers there are. The expected tensor shape for the input to the encoder layer is the same as the output produced; as a result, you can easily stack these layers.

We will see embedding layers in the next chapter. However, you can think of an embedding layer as a dictionary for now. Each entry in the embedding corresponds to each word in a fixed-size vocabulary. Similar words should have similar vectors. The **d_model** hyperparameter specifies the size of the embedding vector. Though you will sometimes preload embeddings from a project such as [Word2vec](https://radimrehurek.com/gensim/models/word2vec.html) or [GloVe](https://nlp.stanford.edu/projects/glove/), the optimizer can train these embeddings with the rest of the transformer. Training your embeddings allows the **d_model** hyperparameter to set to any desired value. If you transfer the embeddings, you must set the **d_model** hyperparameter to the same value as the transferred embeddings.

The **dff** hyperparameter specifies the size of the dense feedforward layers. The **num_heads** hyperparameter sets the number of attention layers heads. Finally, the dropout_rate specifies a dropout percentage to combat overfitting. We discussed dropout previously in this book.

## Inside a Transformer

In this section, we will examine the internals of a transformer so that you become familiar with essential concepts such as:

* Embeddings
* Positional Encoding
* Attention and Self-Attention
* Residual Connection

You can see a lower-level diagram of a transformer in Figure 10.TRANS-2.

**Figure 10.TRANS-2: Architectural Diagram from the Paper**
![Attention is All you Need](https://data.heatonresearch.com/images/jupyter/transformer-2.jpg)

While the original transformer paper is titled "Attention is All you Need," attention isn't the only layer type you need. The transformer also contains dense layers. However, the title "Attention and Dense Layers are All You Need" isn't as catchy.

The transformer begins by tokenizing the input English sentence. Tokens may or may not be words. Generally, familiar parts of words are tokenized and become building blocks of longer words. This tokenization allows common suffixes and prefixes to be understood independently of their stem word. Each token becomes a numeric index that the transformer uses to look up the vector. There are several special tokens:

* Index 0 = Pad
* Index 1 = Unknow
* Index 2 = Start token
* Index 3 = End token

The transformer uses index 0 when we must pad unused space at the end of a tensor. Index 1 is for unknown words. The starting and ending tokens are provided by indexes 2 and 3.

The token vectors are simply the inputs to the attention layers; there is no implied order or position. The transformer adds the slopes of a sine and cosine wave to the token vectors to encode position.

Attention layers have three inputs: key (k), value(v), and query (q). This layer is self-attention if the query, key, and value are the same. The key and value pairs specify the information that the query operates upon. The attention layer learns what positions of data to focus upon.

The transformer presents the position encoded embedding vectors to the first self-attention segment in the encoder layer. The output from the attention is normalized and ultimately becomes the hidden state after all encoder layers are processed.

The hidden state is only calculated once per query. Once the input Spanish sentence becomes a hidden state, this value is presented repeatedly to the decoder until the decoder forms the final Spanish sentence.

This section presented a high-level introduction to transformers. In the next part, we will implement the encoder and apply it to time series. In the following chapter, we will use [Hugging Face](https://huggingface.co/) transformers to perform natural language processing.

## Translation Example




In [ ]:
# Cell #1
!pip -q install datasets

# Cell #2
import math
import re
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Cell #3
from datasets import load_dataset

raw_data = load_dataset(
    "opus_books",
    "en-fr",
    split="train"
)

print(raw_data)
print(raw_data[0])

# Cell #4
# Use a small subset so this remains a teaching example.

MAX_EXAMPLES = 20000
MAX_LEN = 32
VOCAB_SIZE = 10000

pairs = []

for row in raw_data.select(range(min(MAX_EXAMPLES, len(raw_data)))):
    en = row["translation"]["en"]
    fr = row["translation"]["fr"]

    if len(en.split()) <= MAX_LEN and len(fr.split()) <= MAX_LEN:
        pairs.append((en, fr))

print(f"Usable sentence pairs: {len(pairs):,}")
print(pairs[0])

# Cell #5
def tokenize(text):
    text = text.lower().strip()
    return re.findall(r"\w+|[^\w\s]", text, re.UNICODE)


SPECIALS = ["<pad>", "<sos>", "<eos>", "<unk>"]
PAD, SOS, EOS, UNK = range(4)


def build_vocab(texts, max_size):
    counter = Counter()

    for text in texts:
        counter.update(tokenize(text))

    most_common = counter.most_common(max_size - len(SPECIALS))
    vocab = SPECIALS + [word for word, count in most_common]

    stoi = {word: i for i, word in enumerate(vocab)}
    itos = {i: word for word, i in stoi.items()}

    return stoi, itos


src_stoi, src_itos = build_vocab([en for en, fr in pairs], VOCAB_SIZE)
tgt_stoi, tgt_itos = build_vocab([fr for en, fr in pairs], VOCAB_SIZE)

print(f"English vocab size: {len(src_stoi):,}")
print(f"French vocab size: {len(tgt_stoi):,}")





Using device: cuda


README.md: 0.00B [00:00, ?B/s]

en-fr/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/127085 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'translation'],
    num_rows: 127085
})
{'id': '0', 'translation': {'en': 'The Wanderer', 'fr': 'Le grand Meaulnes'}}
Usable sentence pairs: 13,483
('The Wanderer', 'Le grand Meaulnes')
English vocab size: 10,000
French vocab size: 10,000


In [ ]:
# Cell #6
def encode(text, stoi):
    tokens = tokenize(text)
    ids = [SOS]

    for token in tokens:
        ids.append(stoi.get(token, UNK))

    ids.append(EOS)
    return ids


class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        en, fr = self.pairs[idx]

        src = torch.tensor(encode(en, src_stoi), dtype=torch.long)
        tgt = torch.tensor(encode(fr, tgt_stoi), dtype=torch.long)

        return src, tgt


def collate_batch(batch):
    src_batch, tgt_batch = zip(*batch)

    src_batch = nn.utils.rnn.pad_sequence(
        src_batch, batch_first=True, padding_value=PAD
    )

    tgt_batch = nn.utils.rnn.pad_sequence(
        tgt_batch, batch_first=True, padding_value=PAD
    )

    return src_batch, tgt_batch


train_size = int(0.9 * len(pairs))
train_pairs = pairs[:train_size]
valid_pairs = pairs[train_size:]

train_loader = DataLoader(
    TranslationDataset(train_pairs),
    batch_size=64,
    shuffle=True,
    collate_fn=collate_batch,
)

valid_loader = DataLoader(
    TranslationDataset(valid_pairs),
    batch_size=64,
    shuffle=False,
    collate_fn=collate_batch,
)




In [ ]:
# Cell #7
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class TinyTranslator(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=128,
        nhead=4,
        num_layers=2,
        dim_feedforward=512,
        dropout=0.1,
    ):
        super().__init__()

        self.d_model = d_model

        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD)

        self.positional_encoding = PositionalEncoding(d_model)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )

        self.output_layer = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt):
        src_key_padding_mask = src == PAD
        tgt_key_padding_mask = tgt == PAD

        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            tgt.size(1),
            device=tgt.device,
        )

        src_emb = self.src_embedding(src) * math.sqrt(self.d_model)
        tgt_emb = self.tgt_embedding(tgt) * math.sqrt(self.d_model)

        src_emb = self.positional_encoding(src_emb)
        tgt_emb = self.positional_encoding(tgt_emb)

        output = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )

        return self.output_layer(output)


model = TinyTranslator(
    src_vocab_size=len(src_stoi),
    tgt_vocab_size=len(tgt_stoi),
).to(device)

print(model)


TinyTranslator(
  (src_embedding): Embedding(10000, 128, padding_idx=0)
  (tgt_embedding): Embedding(10000, 128, padding_idx=0)
  (positional_encoding): PositionalEncoding()
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=512, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=Tru

In [ ]:
# Cell #8
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)


def train_one_epoch():
    model.train()
    total_loss = 0

    for src, tgt in train_loader:
        src = src.to(device)
        tgt = tgt.to(device)

        decoder_input = tgt[:, :-1]
        expected_output = tgt[:, 1:]

        logits = model(src, decoder_input)

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            expected_output.reshape(-1),
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)


def evaluate():
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for src, tgt in valid_loader:
            src = src.to(device)
            tgt = tgt.to(device)

            decoder_input = tgt[:, :-1]
            expected_output = tgt[:, 1:]

            logits = model(src, decoder_input)

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                expected_output.reshape(-1),
            )

            total_loss += loss.item()

    return total_loss / len(valid_loader)


EPOCHS = 25

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch()
    valid_loss = evaluate()

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Valid Loss: {valid_loss:.4f}"
    )

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 01 | Train Loss: 6.3670 | Valid Loss: 5.2788
Epoch 02 | Train Loss: 5.1196 | Valid Loss: 4.8894
Epoch 03 | Train Loss: 4.7593 | Valid Loss: 4.6793
Epoch 04 | Train Loss: 4.5237 | Valid Loss: 4.5528
Epoch 05 | Train Loss: 4.3555 | Valid Loss: 4.4635
Epoch 06 | Train Loss: 4.2240 | Valid Loss: 4.4086
Epoch 07 | Train Loss: 4.1136 | Valid Loss: 4.3527
Epoch 08 | Train Loss: 4.0218 | Valid Loss: 4.3257
Epoch 09 | Train Loss: 3.9356 | Valid Loss: 4.2890
Epoch 10 | Train Loss: 3.8563 | Valid Loss: 4.2859
Epoch 11 | Train Loss: 3.7868 | Valid Loss: 4.2564
Epoch 12 | Train Loss: 3.7192 | Valid Loss: 4.2379
Epoch 13 | Train Loss: 3.6553 | Valid Loss: 4.2418
Epoch 14 | Train Loss: 3.5918 | Valid Loss: 4.2316
Epoch 15 | Train Loss: 3.5354 | Valid Loss: 4.2289
Epoch 16 | Train Loss: 3.4766 | Valid Loss: 4.2259
Epoch 17 | Train Loss: 3.4242 | Valid Loss: 4.2255
Epoch 18 | Train Loss: 3.3725 | Valid Loss: 4.2143
Epoch 19 | Train Loss: 3.3172 | Valid Loss: 4.2165
Epoch 20 | Train Loss: 3.2648 |

In [ ]:
# Cell #9
def decode(ids, itos):
    words = []

    for idx in ids:
        if idx == EOS:
            break
        if idx not in (PAD, SOS):
            words.append(itos.get(idx, "<unk>"))

    return " ".join(words)


def translate(sentence, max_output_len=40):
    model.eval()

    src = torch.tensor(
        [encode(sentence, src_stoi)],
        dtype=torch.long,
        device=device,
    )

    tgt = torch.tensor([[SOS]], dtype=torch.long, device=device)

    with torch.no_grad():
        for _ in range(max_output_len):
            logits = model(src, tgt)
            next_token = logits[:, -1, :].argmax(dim=-1).item()

            tgt = torch.cat(
                [
                    tgt,
                    torch.tensor([[next_token]], device=device),
                ],
                dim=1,
            )

            if next_token == EOS:
                break

    return decode(tgt[0].tolist(), tgt_itos)

In [ ]:
examples = [
    "I am a man.",
    "She is happy.",
    "He opened the door.",
    "The house is big.",
]

for sentence in examples:
    print(f"EN: {sentence}")
    print(f"FR: {translate(sentence)}")
    print()

EN: I am a man.
FR: je suis un homme charmant .

EN: She is happy.
FR: elle est tres bonne .

EN: He opened the door.
FR: il ouvrit la porte .

EN: The house is big.
FR: le bohémien est là .



# Module 9 Assignment

You can find the fifth assignment here: [assignment 6](https://github.com/jeffheaton/app_deep_learning/blob/main/assignments/assignment_yourname_t81_558_class6.ipynb)
